# ViInfographicVQA — answer-format prompting (single-image task)

Many baseline answers are right in substance but lose all credit on format: `10 ngày` for gold `10`,
a paraphrase or translation instead of the words printed on the image, one item where a list is asked for.
The paper's prompt also asks for "a single term", which works against list answers.

This notebook replaces that prompt with explicit answer-format rules. The rules were written from the
conventions of the **training** answers (e.g. 99% of counting/arithmetic answers are a bare number), not from the test set.

Options: `N_FEWSHOT` adds question/answer examples from the training split; `USE_OCR` also adds the OCR text
cached by notebook 02. The last cell compares every run saved on Drive for this model.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
LIMIT = None            # e.g. 50 for a quick trial run; None = the whole test split
SAVE_TO_DRIVE = True
LOAD_IN_4BIT = False    # set True for the 7B model on a 16 GB GPU

N_FEWSHOT = 0           # 0 = rules only; 8 = also show 8 training question/answer pairs (text only)
USE_OCR = False         # True = also add OCR text (run notebook 02 first, it creates the OCR cache)
OCR_CANVAS = 2560       # must match the value used in notebook 02
MAX_OCR_CHARS = 3000

# Same image budget as the paper's Qwen wrapper.
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1280 * 28 * 28
MAX_NEW_TOKENS = 100

DATASET = "duytranus/ViInfographicVQA"
SPLIT_FILE = "data/single_test.json"
TRAIN_FILE = "data/single_train.json"

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate qwen-vl-utils Levenshtein bitsandbytes

In [ ]:
import json, os, random
from pathlib import Path

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/viinfographicvqa")
else:
    ROOT = Path("/content")
OUT_DIR = ROOT / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_NAME = MODEL_ID.split("/")[-1]
RUN_NAME = "format" + (f"-fewshot{N_FEWSHOT}" if N_FEWSHOT else "") + ("-ocr" if USE_OCR else "")
OUT_FILE = OUT_DIR / f"single_test__{MODEL_NAME}__{RUN_NAME}.json"
BASELINE_FILE = OUT_DIR / f"single_test__{MODEL_NAME}.json"
OCR_FILE = ROOT / "ocr" / f"easyocr_canvas{OCR_CANVAS}.json"
print("results ->", OUT_FILE)

## Data

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download

DATA_DIR = Path("/content/ViInfographicVQA")
hf_hub_download(DATASET, SPLIT_FILE, repo_type="dataset", local_dir=DATA_DIR)
rows = json.loads((DATA_DIR / SPLIT_FILE).read_text(encoding="utf-8"))
if LIMIT:
    rows = rows[:LIMIT]
images = sorted({r["image_path"] for r in rows})
snapshot_download(DATASET, repo_type="dataset", local_dir=DATA_DIR, allow_patterns=images)
missing = [p for p in images if not (DATA_DIR / p).exists()]
assert not missing, f"{len(missing)} images failed to download, e.g. {missing[:3]}"
print(len(rows), "questions on", len(images), "images")

ocr = {}
if USE_OCR:
    assert OCR_FILE.exists(), f"No OCR cache at {OCR_FILE}: run notebook 02 first (same OCR_CANVAS)."
    ocr = json.loads(OCR_FILE.read_text(encoding="utf-8"))
    lacking = [p for p in images if p not in ocr]
    assert not lacking, f"OCR cache lacks {len(lacking)} of the images: re-run notebook 02 with the same LIMIT."

## Prompt

In [ ]:
SYSTEM_PROMPT = """Answer the following question based solely on the image content. Give only the answer, with no explanation, following these rules:
- If the question asks you to count or calculate (how many, total, difference), answer with the number only, without a unit or extra words.
- If the answer is a value shown in the image, copy it exactly as written there, including its unit or symbol (for example % or °C).
- If the answer is text shown in the image, copy the words exactly as they appear. Do not paraphrase or translate them.
- If the question offers options, answer with the chosen option copied exactly from the question.
- If several items are asked for, list all of them, separated by commas."""
OCR_HEADER = "Text extracted from the image by OCR (it may contain errors; trust the image when they disagree):"


def pick_fewshot(train_rows: list, n: int, seed: int = 0) -> list:
    """n short training examples, spread evenly over the four answer types."""
    rng = random.Random(seed)
    by_type = {}
    for r in train_rows:
        if len(r["question"]) <= 160 and len(r["answer"]) <= 60:
            by_type.setdefault(r["answer_source"], []).append(r)
    types = sorted(by_type)
    picked = []
    for i in range(n):
        picked.append(rng.choice(by_type[types[i % len(types)]]))
    return picked


fewshot_block = ""
if N_FEWSHOT:
    hf_hub_download(DATASET, TRAIN_FILE, repo_type="dataset", local_dir=DATA_DIR)
    train_rows = json.loads((DATA_DIR / TRAIN_FILE).read_text(encoding="utf-8"))
    examples = pick_fewshot(train_rows, N_FEWSHOT)
    fewshot_block = ("Examples of the expected answer format (they are about other images):\n"
                     + "\n".join(f"Question: {e['question']}\nAnswer: {e['answer']}" for e in examples) + "\n\n")


def build_messages(question: str, image_path: str, ocr_text: str = "") -> list:
    text = fewshot_block
    if USE_OCR:
        text += f"{OCR_HEADER}\n{ocr_text[:MAX_OCR_CHARS]}\n\n"
    text += f"Question: {question}\nAnswer:"
    return [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "image", "image": f"file://{image_path}"},
                                         {"type": "text", "text": text}]}]


print(SYSTEM_PROMPT, "\n")
print(build_messages(rows[0]["question"], "IMAGE", ocr.get(rows[0]["image_path"], ""))[1]["content"][1]["text"])

## Model

In [ ]:
import torch
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
kwargs = dict(torch_dtype=torch.bfloat16, device_map="cuda")
if LOAD_IN_4BIT:
    kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs).eval()
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, use_fast=True)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 1e9:.1f} GB used")

## Inference

In [ ]:
from tqdm.auto import tqdm


def parse_answer(response: str) -> str:
    return response.split("Answer:")[-1].strip() if "Answer:" in response else response.strip()


@torch.no_grad()
def infer(question: str, image_path: str, ocr_text: str = "") -> str:
    messages = build_messages(question, image_path, ocr_text)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = out[:, inputs.input_ids.shape[1]:]
    return parse_answer(processor.batch_decode(new_tokens, skip_special_tokens=True)[0])


done = {}
if OUT_FILE.exists():
    done = {r["question_id"]: r["predict"] for r in json.loads(OUT_FILE.read_text(encoding="utf-8"))
            if "predict" in r}
    print("resuming:", len(done), "answers already saved")


def save():
    OUT_FILE.write_text(json.dumps(rows, ensure_ascii=False, indent=1), encoding="utf-8")


for i, row in enumerate(tqdm(rows)):
    if row["question_id"] in done:
        row["predict"] = done[row["question_id"]]
        continue
    row["predict"] = infer(row["question"], str(DATA_DIR / row["image_path"]), ocr.get(row["image_path"], ""))
    if i % 25 == 0:
        save()
save()
print("saved", OUT_FILE)

## Score

`ANLS` reproduces the paper's scorer exactly; `ANLS-std` is the textbook definition; `Acc` is exact match.
Paper, single-image overall ANLS: Qwen2.5-VL-7B 67.42 · fine-tuned 67.80 · InternVL3.5-8B 67.02 · Ovis2.5-9B 71.02.

In [ ]:
from collections import defaultdict

import Levenshtein

THRESHOLD = 0.5


def clean_prediction(text: str) -> str:
    return text.lower().strip().rstrip(".").replace('"', "").rstrip(">").lstrip("<")


def clean_answer(text: str) -> str:
    return text.lower().strip()


def paper_anls(answer: str, prediction: str) -> float:
    p = clean_prediction(prediction).replace('"', "").rstrip(".").lower()
    score = Levenshtein.ratio(p, clean_answer(answer))
    return score if score >= THRESHOLD else 0.0


def standard_anls(answer: str, prediction: str) -> float:
    p, g = clean_prediction(prediction), clean_answer(answer)
    if not p and not g:
        return 1.0
    nl = Levenshtein.distance(p, g) / max(len(p), len(g))
    return 1.0 - nl if nl < THRESHOLD else 0.0


def exact_match(answer: str, prediction: str) -> float:
    return float(clean_prediction(prediction) == clean_answer(answer))


METRICS = {"anls": paper_anls, "anls_std": standard_anls, "acc": exact_match}

# Fields to break results down by; whichever exist in the split are used.
GROUP_FIELDS = ("answer_source", "element", "image_type")


def _as_labels(value) -> list[str]:
    """`element` is a string in most rows but a list in a few."""
    if value is None:
        return []
    return [str(v) for v in value] if isinstance(value, list) else [str(value)]


def score(rows: list[dict], pred_key: str = "predict") -> dict:
    """Return {"overall": {...}, "<field>": {"<label>": {...}}} with scores in percent."""
    sums: dict = defaultdict(lambda: defaultdict(lambda: defaultdict(float)))
    for row in rows:
        vals = {name: fn(row["answer"], str(row.get(pred_key, ""))) for name, fn in METRICS.items()}
        groups = [("overall", "overall")]
        groups += [(f, label) for f in GROUP_FIELDS for label in _as_labels(row.get(f))]
        for field, label in groups:
            bucket = sums[field][label]
            bucket["count"] += 1
            for name, v in vals.items():
                bucket[name] += v
    out: dict = {}
    for field, labels in sums.items():
        table = {label: {"count": int(b["count"]),
                         **{m: round(100 * b[m] / b["count"], 2) for m in METRICS}}
                 for label, b in labels.items()}
        out[field] = table["overall"] if field == "overall" else table
    return out


def format_report(result: dict) -> str:
    lines = [f"{'':32s}{'n':>6s}{'ANLS':>8s}{'ANLS-std':>10s}{'Acc':>8s}"]

    def row(name: str, s: dict) -> str:
        return f"{name[:31]:32s}{s['count']:6d}{s['anls']:8.2f}{s['anls_std']:10.2f}{s['acc']:8.2f}"

    lines.append(row("Overall", result["overall"]))
    for field in GROUP_FIELDS:
        if field in result:
            lines.append(f"-- {field}")
            for label, s in sorted(result[field].items(), key=lambda kv: -kv[1]["count"]):
                lines.append(row("  " + label, s))
    return "\n".join(lines)


print(MODEL_ID, "|", RUN_NAME)
print(format_report(score(rows)))

## Compare every run saved for this model (ANLS, on the questions all runs answered)

In [ ]:
runs = {}
for path in sorted(OUT_DIR.glob(f"single_test__{MODEL_NAME}*.json")):
    name = path.stem.split("__")[2] if path.stem.count("__") >= 2 else "baseline"
    runs[name] = {r["question_id"]: r for r in json.loads(path.read_text(encoding="utf-8")) if "predict" in r}
common = set.intersection(*(set(v) for v in runs.values()))
names = sorted(runs, key=lambda n: (n != "baseline", n))
scores = {n: score([runs[n][q] for q in sorted(common)]) for n in names}
print(f"{len(common)} questions answered by all {len(names)} runs\n")
print(f"{'':28s}{'n':>6s}" + "".join(f"{n[:27]:>29s}" for n in names))


def line(label, pick):
    cells = [pick(scores[n]) for n in names]
    print(f"{label[:27]:28s}{cells[0]['count']:6d}" + "".join(f"{c['anls']:29.2f}" for c in cells))


line("Overall", lambda s: s["overall"])
for field in ("answer_source", "element"):
    print("--", field)
    ref = scores[names[0]][field]
    for label in sorted(ref, key=lambda k: -ref[k]["count"]):
        line("  " + label, lambda s, f=field, l=label: s[f][l])

if "baseline" in runs and RUN_NAME in runs:
    def ok(run, q):
        return paper_anls(runs[run][q]["answer"], runs[run][q]["predict"]) > 0
    fixed = [q for q in sorted(common) if not ok("baseline", q) and ok(RUN_NAME, q)]
    broken = [q for q in sorted(common) if ok("baseline", q) and not ok(RUN_NAME, q)]
    print(f"\n{RUN_NAME} vs baseline -> wrong to right: {len(fixed)}   right to wrong: {len(broken)}")
    for title, group in (("FIXED", fixed), ("BROKEN", broken)):
        print(f"\n== {title} (first 8)")
        for q in group[:8]:
            r = runs[RUN_NAME][q]
            print(f"[{r['answer_source']} | {r['element']}] {r['question']}\n   gold: {r['answer']}\n"
                  f"   base: {runs['baseline'][q]['predict']}\n   new : {r['predict']}\n")
else:
    print("\nNo baseline run found: run notebook 01 with the same MODEL_ID to get the fixed/broken lists.")